In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# Download once; subsequent runs read existing disk files.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests images under the split directory.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import run_task
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    PipelineConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import BoVWExtractorConfig
from dataeval_flow.workflows.quality import QualityConfig

cleaning = QualityConfig(
    name="mv_cleaning",
    outliers={"flags": ["dimension", "pixel", "visual"], "outlier_threshold": "adaptive"},
    duplicates={"cluster_algorithm": "hdbscan", "merge_near_duplicates": True},
)

task = TaskConfig(
    name="mv_tuning",
    workflow="mv_cleaning",
    sources="mv_src",
    extractor="bovw_ext",
    matrix={
        "outliers.outlier_threshold": [["adaptive", 2.5], ["adaptive", 3.5], ["adaptive", 4.5]],
        "duplicates.cluster_sensitivity": [0.5, 2.0, 3.0],
    },
)

config = PipelineConfig(
    max_processes=2,
    # Each run starts from this seed, so no run's clustering depends on the runs before it.
    seed=42,
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    views=[
        # Shuffle before limiting to avoid sampling only the first alphabetical classes.
        ViewConfig(
            name="sample1000",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 42}),
                ViewOperation(type="Limit", params={"size": 1000}),
            ],
        ),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train", view="sample1000"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[cleaning],
    tasks=[task],
)

In [3]:
result = run_task(config, task, cache_dir=Path("./cache"))
print(type(result).__name__, len(result.runs), "runs")

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_cache.py:273: UserWarning: `height` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  export(path)
/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/dataeval/_embeddings.py:566: UserWarning: BoVWExtractor is fitting its vocabulary on the data of this one call. Inside Embeddings, Duplicates or Outliers that call is a single batch of batch_size images, not the whole dataset, and every later batch is encoded with that vocabulary. Call fit() on the images you want the vocabulary learned from, then pass the fitted extractor. This warning is not raised once the extractor is fitted.
  embeddings = self._rows(self._extractor(batch_images), len(batch_idx))


MatrixResult 9 runs


In [4]:
print(result.report(detailed=False, width=160))


  QUALITY
  Workflow:  mv_cleaning (quality), matrix of 9 runs
  Timestamp: 2026-11-14T08:53:02.944258+00:00
  Duration:  30.14s
  Source:    mv_src (mv_train[sample1000])
  Model:     bovw_ext (bovw)

  Health: 27 warnings [!!] across 9 runs — review flagged findings

  #  outliers.outlier_threshold  duplicates.cluster_sensitivity  Health  Image Outliers           Class Outliers                  Image Duplicates
  -  --------------------------  ------------------------------  ------  -----------------------  ------------------------------  ------------------------------
  1  [adaptive, 2.5]             0.5                             [!!]    [!!] 157 images (15.7%)  [!!] worst: 2S19 MSTA (25.0%),  [!!] 4 exact (0.4%), 6 near
                                                                                                  23/24 classes over 3.0%         (0.6%)

  2  [adaptive, 2.5]             2.0                             [!!]    [!!] 157 images (15.7%)  [!!] worst: 2S19 MSTA (25.0

In [5]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [6]:
show(result.to_html(detailed=False))

<iframe srcdoc="<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Quality — mv_cleaning</title>
<style>:root { color-scheme: light dark;
 --bg: #ffffff; --card: #ffffff; --soft: #f6f8fa; --ink: #111827; --muted: #6b7280; --rule: #e5e7eb;
 --link: #1d4ed8; --ok: #16a34a; --info: #2563eb; --warning: #d97706; --mark: #dc2626;
 --s0: #2563eb; --s1: #d97706; --s2: #7c3aed; --s3: #16a34a;
 --tag: #eef2ff; --tag-ink: #1e1b4b;
}
@media (prefers-color-scheme: dark) { :root {
 --bg: #0f1115; --card: #171a21; --soft: #1d212a; --ink: #e5e7eb; --muted: #9ca3af; --rule: #2a2f3a;
 --link: #93c5fd; --ok: #22c55e; --info: #60a5fa; --warning: #f59e0b; --mark: #f87171;
 --s0: #60a5fa; --s1: #f59e0b; --s2: #a78bfa; --s3: #22c55e;
 --tag: #232a3d; --tag-ink: #c7d2fe;
} }
body { margin: 0; color: var(--ink); background: var(--bg);
 font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; }
main { max-width: 72rem; margin: 0 auto; padding: 1.5rem; }
a { color: var(--link); }
h1, h2, h3, h4 { margin: 1.25rem 0 0.5rem; line-height: 1.25; }
h1 { font-size: 1.6rem; } h2 { font-size: 1.2rem; } h3 { font-size: 1.05rem; } h4 { font-size: 0.95rem; }
nav.contents { border: 1px solid var(--rule); border-radius: 7px; padding: 0.25rem 1rem; margin-bottom: 1.5rem; }
nav.contents h2 { font-size: 1rem; margin: 0.5rem 0; }
.controls { display: flex; justify-content: flex-end; gap: 0.5rem; margin-bottom: 0.5rem; }
.controls button { font: inherit; font-size: 0.8rem; padding: 0.2rem 0.7rem; cursor: pointer;
 color: var(--ink); background: var(--soft); border: 1px solid var(--rule); border-radius: 4px; }
.report { margin-bottom: 2.5rem; }
.report-head { display: flex; align-items: baseline; gap: 0.75rem; flex-wrap: wrap;
 border-bottom: 2px solid var(--ink); padding-bottom: 0.3rem; }
.report-head h1 { margin: 0; }
dl.fields.provenance { display: grid; grid-template-columns: max-content 1fr; gap: 0.15rem 1rem;
 margin: 0.6rem 0 1.2rem; padding: 0.5rem 0.8rem; background: var(--soft); border-radius: 6px;
 color: var(--muted); font-size: 0.85rem; }
dl.provenance dt { font-weight: 600; color: var(--ink); }
dl.provenance dd { margin: 0; overflow-wrap: anywhere; }
section.section > h2 { border-bottom: 1px solid var(--rule); padding-bottom: 0.2rem; }
details.card, details.panel { background: var(--card); border: 1px solid var(--rule);
 border-left: 4px solid var(--info); border-radius: 7px; padding: 0.3rem 1rem 0.6rem; margin: 0.9rem 0; }
details.card.ok { border-left-color: var(--ok); } details.card.warning { border-left-color: var(--warning); }
details.panel { border-left-color: var(--rule); margin-top: 1.5rem; }
:is(details.card, details.panel) > summary { cursor: pointer; list-style: none; }
:is(details.card, details.panel) > summary::-webkit-details-marker { display: none; }
:is(details.card, details.panel) > summary h2 { display: inline; font-size: 1.1rem; margin: 0; }
:is(details.card, details.panel) > summary::before { content: "▸"; color: var(--muted); margin-right: 0.4rem; }
:is(details.card, details.panel)[open] > summary::before { content: "▾"; }
.brief { font-weight: normal; color: var(--muted); margin-left: 0.5rem; }
.badge { font-size: 0.72rem; font-weight: 700; letter-spacing: 0.02em; border-radius: 1rem; padding: 0.05rem 0.55rem;
 margin-left: 0.5rem; color: #fff; background: var(--info); vertical-align: middle; }
.badge.ok { background: var(--ok); } .badge.warning { background: var(--warning); }
.badge.failed { background: var(--mark); }
code, pre { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "DejaVu Sans Mono", monospace; }
code { font-size: 0.9em; background: var(--soft); padding: 0 0.2em; border-radius: 3px; }
pre { background: var(--soft); padding: 0.75rem; overflow-x: auto; font-size: 0.85rem; }
pre code { background: none; padding: 0; }
table { border-collapse: collapse; margin: 0.5rem 0; font-size: 0.85rem;

In [7]:
run = result.runs[3]
print(run.number, run.label)
for finding in run.result.findings:
    print(f"{finding.severity:<8} {finding.title:<20} {finding.brief}")

4 outliers.outlier_threshold=[adaptive, 3.5], duplicates.cluster_sensitivity=0.5
warning  Image Outliers       112 images (11.2%)
warning  Class Outliers       worst: Tornado (20.4%), 23/24 classes over 3.0%
warning  Image Duplicates     4 exact (0.4%), 6 near (0.6%)


In [8]:
print(run.result.steps["duplicates"].output.data())

shape: (5, 7)
┌──────────┬───────┬──────────┬──────────────┬─────────────┬────────────┬───────────────────────┐
│ group_id ┆ level ┆ dup_type ┆ item_indices ┆ methods     ┆ differs_on ┆ annotation_divergence │
│ ---      ┆ ---   ┆ ---      ┆ ---          ┆ ---         ┆ ---        ┆ ---                   │
│ i64      ┆ str   ┆ str      ┆ list[i64]    ┆ list[str]   ┆ list[str]  ┆ f64                   │
╞══════════╪═══════╪══════════╪══════════════╪═════════════╪════════════╪═══════════════════════╡
│ 0        ┆ item  ┆ exact    ┆ [337, 344]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 1        ┆ item  ┆ exact    ┆ [496, 721]   ┆ ["xxhash"]  ┆ null       ┆ null                  │
│ 2        ┆ item  ┆ near     ┆ [253, 855]   ┆ ["dhash",   ┆ null       ┆ null                  │
│          ┆       ┆          ┆              ┆ "phash"]    ┆            ┆                       │
│ 3        ┆ item  ┆ near     ┆ [337, 344]   ┆ ["cluster"] ┆ null       ┆ null                  │
│ 4   